# Beach Human Detection — Faster R-CNN ResNet-50 FPN v2 (100 epochs)

Downloads Roboflow workspace `image-analysis-project-rfnyq`, project `vid_ds_bhd`, version 1, using the YOLO-format export **only as an annotation format**. The trained model is Faster R-CNN, not YOLO.

Classes: `Human on Land`, `Human in Water`. Model: COCO-pretrained Faster R-CNN ResNet-50 FPN v2.

**Baseline settings from the earlier experiment:** batch size 4; SGD lr 0.005, momentum 0.9, weight decay 0.0005; StepLR step size 4, gamma 0.1. Previously recorded metrics: mAP@0.5 = 0.8882 and mAP@0.5:0.95 = 0.6661 (from the older dataset/experiment). This notebook uses 100 epochs as requested. Results on the new video-frame dataset will not be directly comparable unless the split and annotations are comparable.

**API-key safety:** the key pasted into chat should be revoked/regenerated in Roboflow. Enter the new key at runtime when prompted; it is not saved in this notebook.

## 1. Colab setup
In Colab choose **Runtime → Change runtime type → GPU**, then run cells in order.

In [ ]:
!pip -q install roboflow pyyaml scikit-learn seaborn pandas matplotlib tqdm

In [ ]:
import os, json, time, random, getpass, shutil
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, yaml
from tqdm.auto import tqdm
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type != "cuda": print("Enable a GPU runtime for practical 100-epoch training.")

## 2. Download Roboflow dataset

In [ ]:
from roboflow import Roboflow
ROBOFLOW_API_KEY = getpass.getpass("Enter your NEW Roboflow API key: ")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("image-analysis-project-rfnyq").project("vid_ds_bhd")
version = project.version(1)
dataset = version.download("yolov8")
DATASET_DIR = Path(dataset.location)
print("Dataset:", DATASET_DIR)
print("Contents:", [p.name for p in DATASET_DIR.iterdir()])

## 3. Read class names and inspect splits

In [ ]:
yaml_path = DATASET_DIR / "data.yaml"
if not yaml_path.exists():
    found = list(DATASET_DIR.rglob("data.yaml"))
    if not found: raise FileNotFoundError("data.yaml was not found in the Roboflow download.")
    yaml_path = found[0]
cfg = yaml.safe_load(yaml_path.read_text(encoding="utf-8"))
print(json.dumps(cfg, indent=2))
CLASS_NAMES = cfg.get("names", ["Human on Land", "Human in Water"])
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = [CLASS_NAMES[k] for k in sorted(CLASS_NAMES, key=lambda x: int(x))]
CLASS_NAMES = list(CLASS_NAMES)
if len(CLASS_NAMES) != 2:
    raise ValueError(f"Expected 2 classes, found {CLASS_NAMES}. Check Roboflow project classes.")
print("Annotation IDs:", {i:n for i,n in enumerate(CLASS_NAMES)})
SPLITS = {s: DATASET_DIR/s for s in ["train","valid","test"]}
for s,p in SPLITS.items(): print(s, p.exists(), p)
if not SPLITS["train"].exists() or not SPLITS["valid"].exists():
    raise FileNotFoundError("Expected train/ and valid/ directories in the Roboflow export.")

## 4. YOLO-label dataset loader for Faster R-CNN

In [ ]:
IMAGE_EXTS = {".jpg",".jpeg",".png",".bmp",".webp"}
class RoboflowDataset(Dataset):
    def __init__(self, split_dir):
        self.root = Path(split_dir)
        self.img_dir, self.lab_dir = self.root/"images", self.root/"labels"
        if not self.img_dir.exists(): raise FileNotFoundError(self.img_dir)
        self.images = sorted(p for p in self.img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
        if not self.images: raise RuntimeError(f"No images found in {self.img_dir}")
    def __len__(self): return len(self.images)
    def __getitem__(self, i):
        p = self.images[i]
        im = Image.open(p).convert("RGB")
        w,h = im.size
        x = TF.to_tensor(im)
        boxes, labels = [], []
        lp = self.lab_dir/(p.stem+".txt")
        if lp.exists():
            for line in lp.read_text(encoding="utf-8").splitlines():
                a = line.split()
                if len(a) < 5: continue
                c,xc,yc,bw,bh = int(float(a[0])), *map(float,a[1:5])
                if not 0 <= c < len(CLASS_NAMES): continue
                x1=max(0,(xc-bw/2)*w); y1=max(0,(yc-bh/2)*h)
                x2=min(w,(xc+bw/2)*w); y2=min(h,(yc+bh/2)*h)
                if x2>x1 and y2>y1:
                    boxes.append([x1,y1,x2,y2]); labels.append(c+1)
        target = {
            "boxes":torch.tensor(boxes,dtype=torch.float32).reshape(-1,4),
            "labels":torch.tensor(labels,dtype=torch.int64),
            "image_id":torch.tensor([i]),
            "area":torch.tensor([(b[2]-b[0])*(b[3]-b[1]) for b in boxes],dtype=torch.float32),
            "iscrowd":torch.zeros(len(boxes),dtype=torch.int64)
        }
        return x,target

def collate_fn(batch): return tuple(zip(*batch))
train_ds, valid_ds = RoboflowDataset(SPLITS["train"]), RoboflowDataset(SPLITS["valid"])
test_ds = RoboflowDataset(SPLITS["test"]) if SPLITS["test"].exists() and (SPLITS["test"]/"images").exists() else None
BATCH_SIZE, NUM_WORKERS = 4, 2
train_loader = DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,collate_fn=collate_fn,pin_memory=DEVICE.type=="cuda")
valid_loader = DataLoader(valid_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,collate_fn=collate_fn,pin_memory=DEVICE.type=="cuda")
test_loader = DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,collate_fn=collate_fn,pin_memory=DEVICE.type=="cuda") if test_ds else None
print("Images — train:",len(train_ds),"valid:",len(valid_ds),"test:",len(test_ds) if test_ds else 0)

## 5. Visual check of converted ground-truth boxes

In [ ]:
def show_gt(ds, i=0):
    im,t=ds[i]; fig,ax=plt.subplots(figsize=(10,6)); ax.imshow(im.permute(1,2,0))
    for b,l in zip(t["boxes"].numpy(),t["labels"].numpy()):
        x1,y1,x2,y2=b; ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,linewidth=2))
        ax.text(x1,max(0,y1-3),CLASS_NAMES[l-1],color="yellow",bbox=dict(facecolor="black",alpha=.6))
    ax.set_title(train_ds.images[i].name); ax.axis("off"); plt.show()
for i in range(min(3,len(train_ds))): show_gt(train_ds,i)

## 6. Model and training settings — 100 epochs

In [ ]:
model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, len(CLASS_NAMES)+1)
model.to(DEVICE)

NUM_EPOCHS = 100
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.1)

try:
    from google.colab import drive
    IN_COLAB=True
except Exception: IN_COLAB=False
if IN_COLAB:
    drive.mount("/content/drive")
    OUT=Path("/content/drive/MyDrive/beach_human_detection_outputs")
else: OUT=Path("/content/beach_human_detection_outputs")
OUT.mkdir(parents=True,exist_ok=True)
BEST_PATH, LAST_PATH = OUT/"fasterrcnn_best.pth", OUT/"fasterrcnn_last.pth"
HISTORY_PATH=OUT/"training_history.csv"
print("Saving results to",OUT)

## 7. Train and calculate validation metrics

The validation evaluator uses greedy one-to-one box matching at IoU ≥ 0.5 and computes per-class AP from ranked detections. It is a lightweight project metric implementation, not the official COCO evaluator. It is consistent within this notebook, but small differences from earlier evaluation code are possible.

In [ ]:
def iou_one(box, boxes):
    if len(boxes)==0: return np.zeros(0)
    xx1=np.maximum(box[0],boxes[:,0]); yy1=np.maximum(box[1],boxes[:,1])
    xx2=np.minimum(box[2],boxes[:,2]); yy2=np.minimum(box[3],boxes[:,3])
    inter=np.maximum(0,xx2-xx1)*np.maximum(0,yy2-yy1)
    a=max(0,box[2]-box[0])*max(0,box[3]-box[1])
    b=np.maximum(0,boxes[:,2]-boxes[:,0])*np.maximum(0,boxes[:,3]-boxes[:,1])
    return inter/np.maximum(a+b-inter,1e-9)

@torch.no_grad()
def get_records(model, loader):
    model.eval(); records=[]
    for imgs, targets in tqdm(loader,leave=False):
        outs=model([im.to(DEVICE) for im in imgs])
        for im,t,o in zip(imgs,targets,outs):
            records.append({"image":im,
                "gb":t["boxes"].numpy(),"gl":t["labels"].numpy(),
                "pb":o["boxes"].cpu().numpy(),"pl":o["labels"].cpu().numpy(),
                "ps":o["scores"].cpu().numpy()})
    return records

def evaluate_records(records, iou_thr=.5, score_thr=.5):
    per={}; sumtp=sumfp=sumfn=0
    for cls,name in enumerate(CLASS_NAMES,1):
        gtsets=[{"b":r["gb"][r["gl"]==cls],"m":np.zeros(np.sum(r["gl"]==cls),bool)} for r in records]
        ngt=sum(len(g["b"]) for g in gtsets); det=[]
        for j,r in enumerate(records):
            mask=(r["pl"]==cls)&(r["ps"]>=.001)
            det += [(float(s),j,b) for b,s in zip(r["pb"][mask],r["ps"][mask])]
        det.sort(key=lambda z:z[0],reverse=True)
        tp,fp=[],[]
        for score,j,b in det:
            g=gtsets[j]; ov=iou_one(b,g["b"])
            if len(ov):
                ov[g["m"]]=-1; k=int(np.argmax(ov))
                if ov[k]>=iou_thr: tp.append(1);fp.append(0);g["m"][k]=True
                else: tp.append(0);fp.append(1)
            else: tp.append(0);fp.append(1)
        if tp:
            ct=np.cumsum(tp); cf=np.cumsum(fp); rec=ct/max(ngt,1); prec=ct/np.maximum(ct+cf,1)
            mr=np.r_[0,rec,1]; mp=np.r_[0,prec,0]
            for k in range(len(mp)-2,-1,-1): mp[k]=max(mp[k],mp[k+1])
            inds=np.where(mr[1:]!=mr[:-1])[0]; ap=float(np.sum((mr[inds+1]-mr[inds])*mp[inds+1]))
        else: ap=0.; rec=np.array([]);prec=np.array([])
        ctp=cfp=cfn=0
        for r in records:
            gb=r["gb"][r["gl"]==cls]; mask=(r["pl"]==cls)&(r["ps"]>=score_thr)
            pb=r["pb"][mask]; matched=np.zeros(len(gb),bool)
            for b in pb:
                ov=iou_one(b,gb)
                if len(ov):
                    ov[matched]=-1;k=int(np.argmax(ov))
                    if ov[k]>=iou_thr: ctp+=1;matched[k]=True
                    else: cfp+=1
                else: cfp+=1
            cfn+=int((~matched).sum())
        p=ctp/max(ctp+cfp,1); rr=ctp/max(ctp+cfn,1); f=2*p*rr/max(p+rr,1e-12)
        per[name]={"AP":ap,"precision":p,"recall":rr,"F1":f,"ground_truth_count":int(ngt)}
        sumtp+=ctp;sumfp+=cfp;sumfn+=cfn
    p=sumtp/max(sumtp+sumfp,1); r=sumtp/max(sumtp+sumfn,1)
    return {"map50":float(np.mean([v["AP"] for v in per.values()])),
            "precision":p,"recall":r,"f1":2*p*r/max(p+r,1e-12),"per_class":per}

def train_epoch():
    model.train(); total=0.; n=0
    for imgs,targets in tqdm(train_loader,leave=False):
        imgs=[x.to(DEVICE) for x in imgs]
        targets=[{k:v.to(DEVICE) for k,v in t.items()} for t in targets]
        loss_dict=model(imgs,targets); loss=sum(loss_dict.values())
        if not torch.isfinite(loss): raise RuntimeError(f"Non-finite loss: {loss.item()}")
        optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        total+=loss.item()*len(imgs);n+=len(imgs)
    return total/max(n,1)

history=[]; best=-1; start=time.time()
for epoch in range(1,NUM_EPOCHS+1):
    t=time.time(); loss=train_epoch(); scheduler.step()
    recs=get_records(model,valid_loader); m=evaluate_records(recs)
    row={"epoch":epoch,"train_loss":loss,"val_map50":m["map50"],"val_precision":m["precision"],
         "val_recall":m["recall"],"val_f1":m["f1"],"lr":optimizer.param_groups[0]["lr"],
         "epoch_seconds":time.time()-t}
    history.append(row); pd.DataFrame(history).to_csv(HISTORY_PATH,index=False)
    state={"epoch":epoch,"model_state_dict":model.state_dict(),"optimizer_state_dict":optimizer.state_dict(),
           "class_names":CLASS_NAMES,"history":history}
    torch.save(state,LAST_PATH)
    if m["map50"]>best: best=m["map50"];torch.save(state,BEST_PATH)
    print(f"Epoch {epoch:03d}/{NUM_EPOCHS} loss={loss:.4f} mAP50={m['map50']:.4f} "
          f"P={m['precision']:.4f} R={m['recall']:.4f} F1={m['f1']:.4f} "
          f"({row['epoch_seconds']:.1f}s)")
print(f"Training time: {(time.time()-start)/60:.1f} minutes")

## 8. Training graphs

In [ ]:
hist=pd.read_csv(HISTORY_PATH)
def savefig(name):
    plt.tight_layout(); plt.savefig(OUT/name,dpi=180,bbox_inches="tight"); plt.show()
plots=[("train_loss","Training Loss","Loss","training_loss.png"),
       ("val_map50","Validation mAP@0.5","mAP@0.5","validation_map50.png"),
       ("val_precision","Validation Precision","Precision","validation_precision.png"),
       ("val_recall","Validation Recall","Recall","validation_recall.png"),
       ("val_f1","Validation F1","F1","validation_f1.png"),
       ("lr","Learning Rate","Learning rate","learning_rate.png")]
for col,title,ylabel,fn in plots:
    plt.figure(figsize=(8,4.5));plt.plot(hist.epoch,hist[col],marker=".",markersize=3)
    plt.title(title);plt.xlabel("Epoch");plt.ylabel(ylabel);plt.grid(True);savefig(fn)
display(hist.tail())

## 9. Final evaluation on held-out test split

In [ ]:
ckpt=torch.load(BEST_PATH,map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"]);model.to(DEVICE).eval()
eval_loader=test_loader if test_loader is not None else valid_loader
eval_split="test" if test_loader is not None else "validation (test split absent)"
print("Evaluation split:",eval_split)
records=get_records(model,eval_loader)
m50=evaluate_records(records,.5,.5)
aps={}
for thr in np.arange(.50,.951,.05):
    aps[round(float(thr),2)]=evaluate_records(records,float(thr),.5)["map50"]
map5095=float(np.mean(list(aps.values())))
print(f"mAP@0.5: {m50['map50']:.4f}")
print(f"mAP@0.5:0.95 (custom average): {map5095:.4f}")
print(f"Precision: {m50['precision']:.4f} | Recall: {m50['recall']:.4f} | F1: {m50['f1']:.4f}")
display(pd.DataFrame(m50["per_class"]).T)
summary={"evaluation_split":eval_split,"epochs":NUM_EPOCHS,"map50":m50["map50"],
         "map50_95_custom":map5095,"precision":m50["precision"],"recall":m50["recall"],
         "f1":m50["f1"],"per_class":m50["per_class"],"ap_by_iou_threshold":aps}
(OUT/"evaluation_summary.json").write_text(json.dumps(summary,indent=2))

## 10. Detection confusion matrix

In [ ]:
def make_cm(records, score_thr=.5, iou_thr=.5):
    # Rows=ground truth, columns=prediction; final category is background.
    n=len(CLASS_NAMES)+1; bg=len(CLASS_NAMES); cm=np.zeros((n,n),int)
    for r in records:
        gb,gl=r["gb"],r["gl"]
        keep=r["ps"]>=score_thr; pb,pl,ps=r["pb"][keep],r["pl"][keep],r["ps"][keep]
        order=np.argsort(-ps);pb,pl=pb[order],pl[order]
        gm=np.zeros(len(gb),bool);pm=np.zeros(len(pb),bool)
        for i,b in enumerate(pb):
            if not len(gb): break
            ov=iou_one(b,gb);ov[gm]=-1;j=int(np.argmax(ov))
            if ov[j]>=iou_thr:
                cm[int(gl[j])-1,int(pl[i])-1]+=1;gm[j]=True;pm[i]=True
        for j in range(len(gb)):
            if not gm[j]: cm[int(gl[j])-1,bg]+=1
        for i in range(len(pb)):
            if not pm[i]: cm[bg,int(pl[i])-1]+=1
    return cm,list(CLASS_NAMES)+["Background"]
cm,labels=make_cm(records)
plt.figure(figsize=(7,6));sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=labels,yticklabels=labels)
plt.xlabel("Predicted");plt.ylabel("Ground truth");plt.title("Detection Confusion Matrix (IoU ≥ 0.5, confidence ≥ 0.5)")
savefig("confusion_matrix.png")
pd.DataFrame(cm,index=labels,columns=labels).to_csv(OUT/"confusion_matrix.csv")

## 11. Precision–Recall, confidence curves and per-class AP

In [ ]:
plt.figure(figsize=(7,5))
for cls,name in enumerate(CLASS_NAMES,1):
    gts=[r["gb"][r["gl"]==cls] for r in records]
    matched=[np.zeros(len(g),bool) for g in gts]
    n_gt=sum(len(g) for g in gts); det=[]
    for j,r in enumerate(records):
        mask=(r["pl"]==cls)&(r["ps"]>=.001)
        det += [(float(s),j,b) for b,s in zip(r["pb"][mask],r["ps"][mask])]
    det.sort(key=lambda x:x[0],reverse=True);tp=[];fp=[]
    for s,j,b in det:
        ov=iou_one(b,gts[j])
        if len(ov):
            ov[matched[j]]=-1;k=int(np.argmax(ov))
            if ov[k]>=.5:tp.append(1);fp.append(0);matched[j][k]=True
            else:tp.append(0);fp.append(1)
        else:tp.append(0);fp.append(1)
    if tp:
        ct=np.cumsum(tp);cf=np.cumsum(fp);rec=ct/max(n_gt,1);pre=ct/np.maximum(ct+cf,1)
        plt.plot(rec,pre,label=name)
plt.xlabel("Recall");plt.ylabel("Precision");plt.title("Precision–Recall Curve (IoU ≥ 0.5)");plt.grid(True);plt.legend();savefig("precision_recall_curve.png")

confs=np.linspace(.05,.95,19); ps=[];rs=[];fs=[]
for c in confs:
    m=evaluate_records(records,.5,float(c));ps.append(m["precision"]);rs.append(m["recall"]);fs.append(m["f1"])
plt.figure(figsize=(8,4.5));plt.plot(confs,ps,label="Precision");plt.plot(confs,rs,label="Recall");plt.plot(confs,fs,label="F1")
plt.xlabel("Confidence threshold");plt.ylabel("Score");plt.title("Precision / Recall / F1 vs Confidence");plt.grid(True);plt.legend();savefig("confidence_curves.png")

per=pd.DataFrame(m50["per_class"]).T
plt.figure(figsize=(7,4.5));plt.bar(per.index,per["AP"]);plt.ylim(0,1);plt.ylabel("AP@0.5");plt.title("Per-Class AP@0.5");plt.grid(axis="y");savefig("per_class_ap.png")

## 12. View sample predictions

In [ ]:
def show_pred(r,thr=.5):
    fig,ax=plt.subplots(figsize=(10,6));ax.imshow(r["image"].permute(1,2,0).numpy())
    for b,s,l in zip(r["pb"],r["ps"],r["pl"]):
        if s<thr:continue
        x1,y1,x2,y2=b;ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,linewidth=2))
        ax.text(x1,max(0,y1-3),f"{CLASS_NAMES[int(l)-1]} {s:.2f}",color="yellow",bbox=dict(facecolor="black",alpha=.6))
    ax.axis("off");plt.show()
for i in range(min(5,len(records))):show_pred(records[i])

## 13. Package results

In [ ]:
zip_path=shutil.make_archive(str(OUT),"zip",root_dir=OUT)
print("Results archive:",zip_path)
print("Saved files:")
for p in sorted(OUT.iterdir()): print(p.name)